# 08 - Refresh Lakebase synced tables
Re-snapshots every synced table listed in `config/lakebase_synced_tables.json` (created once by `scripts/setup_lakebase.sh`) so the app's
Postgres copy matches the freshly scored gold tables. Runs after scoring; waits for all refreshes and fails if any sync fails.

In [ ]:
import json, os, time
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
cfg = json.load(open(os.path.join(os.getcwd(), "..", "..", "config", "lakebase_synced_tables.json")))
names = [f"{cfg['lakebase_catalog']}.{cfg['postgres_schema']}.{t['source']}" for t in cfg["tables"]]

def synced(name):
    return w.api_client.do("GET", f"/api/2.0/postgres/synced_tables/{name}")

updates, missing = {}, []
for n in names:
    try:
        pid = synced(n)["status"]["pipeline_id"]
    except Exception as e:
        missing.append(n); continue
    updates[n] = (pid, w.pipelines.start_update(pid, full_refresh=False).update_id)
print(f"started {len(updates)} refreshes; missing (run scripts/setup_lakebase.sh): {missing}")

In [ ]:
pending, failed = dict(updates), []
deadline = time.time() + 3600
while pending and time.time() < deadline:
    for n, (pid, uid) in list(pending.items()):
        state = w.pipelines.get_update(pid, uid).update.state.value
        if state in ("COMPLETED", "FAILED", "CANCELED"):
            pending.pop(n)
            if state != "COMPLETED": failed.append((n, state))
    time.sleep(20)
assert not pending, f"timed out waiting for: {list(pending)}"
assert not failed, f"sync failures: {failed}"
assert not missing, f"synced tables not set up: {missing}"
print("all synced tables refreshed")